# Phishing Email Detection Model Training

This notebook trains a machine learning model to detect phishing emails using the HuggingFace dataset.

**Dataset:** [zefang-liu/phishing-email-dataset](https://huggingface.co/datasets/zefang-liu/phishing-email-dataset)
- 18,650 email samples
- Labels: "Phishing Email" or "Safe Email"

**Pipeline:**
1. Load dataset from HuggingFace
2. Preprocess text (clean, normalize)
3. TF-IDF Vectorization
4. Train Random Forest Classifier
5. Evaluate and save model

In [1]:
# Install required packages (run this first in Colab)
!pip install datasets nltk scikit-learn joblib -q

In [2]:
# Imports
import pandas as pd
import numpy as np
import re
import string
import joblib
import os
import nltk
from datasets import load_dataset
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

print("All imports successful!")

All imports successful!


In [3]:
# Load Dataset from HuggingFace
print("Loading dataset from HuggingFace...")
dataset = load_dataset("zefang-liu/phishing-email-dataset", split="train")

# Convert to DataFrame
df = pd.DataFrame(dataset)
print(f"Dataset loaded successfully!")
print(f"Total samples: {len(df):,}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nLabel Distribution:")
print(df['Email Type'].value_counts())
df.head()

Loading dataset from HuggingFace...


README.md:   0%|          | 0.00/616 [00:00<?, ?B/s]

Phishing_Email.csv:   0%|          | 0.00/52.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/18650 [00:00<?, ? examples/s]

Dataset loaded successfully!
Total samples: 18,650
Columns: ['Unnamed: 0', 'Email Text', 'Email Type']

Label Distribution:
Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64


,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email


In [4]:
# Text Preprocessing
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Comprehensive text cleaning for email content.
    """
    if not isinstance(text, str):
        return ""
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', ' urltoken ', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', ' emailtoken ', text)
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove special characters but keep spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize and lemmatize
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) > 2]
    
    return ' '.join(words)

# Apply preprocessing
print("Preprocessing text... (this may take a few minutes)")
df['clean_text'] = df['Email Text'].apply(preprocess_text)

# Create binary labels: 1 = Phishing, 0 = Safe
df['label'] = (df['Email Type'] == 'Phishing Email').astype(int)

print(f"Preprocessing complete!")
print(f"\nSample original vs cleaned:")
print(f"Original: {df['Email Text'].iloc[0][:200]}...")
print(f"Cleaned: {df['clean_text'].iloc[0][:200]}...")

Preprocessing text... (this may take a few minutes)
Preprocessing complete!

Sample original vs cleaned:
Original: re : 6 . 1100 , disc : uniformitarianism , re : 1086 ; sex / lang dick hudson 's observations on us use of 's on ' but not 'd aughter ' as a vocative are very thought-provoking , but i am not sure tha...
Cleaned: disc uniformitarianism sex lang dick hudson observation use aughter vocative thought provoking sure fair attribute son treated like senior relative one thing normally use brother way aughter hard imag...
Preprocessing complete!

Sample original vs cleaned:
Original: re : 6 . 1100 , disc : uniformitarianism , re : 1086 ; sex / lang dick hudson 's observations on us use of 's on ' but not 'd aughter ' as a vocative are very thought-provoking , but i am not sure tha...
Cleaned: disc uniformitarianism sex lang dick hudson observation use aughter vocative thought provoking sure fair attribute son treated like senior relative one thing normally use brother way aughter h

In [5]:
# Train-Test Split
X = df['clean_text']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42,
    stratify=y  # Maintain label distribution
)

print(f"Training set: {len(X_train):,} samples")
print(f"Test set: {len(X_test):,} samples")
print(f"\nTraining label distribution:")
print(y_train.value_counts())

Training set: 14,920 samples
Test set: 3,730 samples

Training label distribution:
label
0    9058
1    5862
Name: count, dtype: int64


In [6]:
# Create and Train the Model Pipeline
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=10000,
        ngram_range=(1, 2),  # Unigrams and bigrams
        min_df=2,
        max_df=0.95,
        stop_words='english'
    )),
    ('clf', RandomForestClassifier(
        n_estimators=200,
        max_depth=50,
        min_samples_split=5,
        min_samples_leaf=2,
        random_state=42,
        n_jobs=-1,
        class_weight='balanced'
    ))
])

print("Training model... (this may take a few minutes)")
pipeline.fit(X_train, y_train)
print("Training complete!")

Training model... (this may take a few minutes)
Training complete!
Training complete!


In [7]:
# Model Evaluation
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

print("=" * 50)
print("MODEL EVALUATION RESULTS")
print("=" * 50)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"\n📊 Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

# ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_pred_proba)
print(f"📈 ROC-AUC Score: {roc_auc:.4f}")

# Classification Report
print("\n📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Safe Email', 'Phishing Email']))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print("🔢 Confusion Matrix:")
print(f"   True Negatives (Safe→Safe): {cm[0][0]}")
print(f"   False Positives (Safe→Phishing): {cm[0][1]}")
print(f"   False Negatives (Phishing→Safe): {cm[1][0]}")
print(f"   True Positives (Phishing→Phishing): {cm[1][1]}")

MODEL EVALUATION RESULTS

📊 Accuracy: 0.9485 (94.85%)
📈 ROC-AUC Score: 0.9912

📋 Classification Report:
                precision    recall  f1-score   support

    Safe Email       0.99      0.93      0.96      2264
Phishing Email       0.90      0.98      0.94      1466

      accuracy                           0.95      3730
     macro avg       0.94      0.95      0.95      3730
  weighted avg       0.95      0.95      0.95      3730

🔢 Confusion Matrix:
   True Negatives (Safe→Safe): 2102
   False Positives (Safe→Phishing): 162
   False Negatives (Phishing→Safe): 30
   True Positives (Phishing→Phishing): 1436


In [8]:
# Cross-Validation Score
print("Running 5-fold cross-validation...")
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy', n_jobs=-1)
print(f"\n✅ Cross-Validation Scores: {cv_scores}")
print(f"✅ Mean CV Accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

Running 5-fold cross-validation...

✅ Cross-Validation Scores: [0.9480563  0.95878016 0.95241287 0.94504021 0.95274799]
✅ Mean CV Accuracy: 0.9514 (+/- 0.0093)

✅ Cross-Validation Scores: [0.9480563  0.95878016 0.95241287 0.94504021 0.95274799]
✅ Mean CV Accuracy: 0.9514 (+/- 0.0093)


In [ ]:
# Test with Sample Emails
test_emails = [
    "URGENT: Your account has been compromised! Click here immediately to verify your identity or your account will be suspended.",
    "Hi team, please find attached the quarterly report. Let me know if you have any questions.",
    "Congratulations! You've won $1,000,000! Click now to claim your prize before it expires!",
    "Meeting reminder: Project sync at 3pm tomorrow in Conference Room B.",
    "Your PayPal account has unusual activity. Verify now: http://paypa1-secure.com/verify"
]

print("=" * 60)
print("SAMPLE PREDICTIONS")
print("=" * 60)

for i, email in enumerate(test_emails, 1):
    clean_email = preprocess_text(email)
    prediction = pipeline.predict([clean_email])[0]
    confidence = pipeline.predict_proba([clean_email])[0]
    
    label = "🚨 PHISHING" if prediction == 1 else "✅ SAFE"
    conf_score = confidence[1] if prediction == 1 else confidence[0]
    
    print(f"\n[Email {i}]")
    print(f"Text: {email[:80]}...")
    print(f"Prediction: {label} (Confidence: {conf_score*100:.1f}%)")

SAMPLE PREDICTIONS

[Email 1]
Text: URGENT: Your account has been compromised! Click here immediately to verify your...
Prediction: 🚨 PHISHING (Confidence: 82.6%)

[Email 2]
Text: Hi team, please find attached the quarterly report. Let me know if you have any ...
Prediction: ✅ SAFE (Confidence: 84.8%)

[Email 3]
Text: Congratulations! You've won $1,000,000! Click now to claim your prize before it ...
Prediction: 🚨 PHISHING (Confidence: 80.2%)

[Email 2]
Text: Hi team, please find attached the quarterly report. Let me know if you have any ...
Prediction: ✅ SAFE (Confidence: 84.8%)

[Email 3]
Text: Congratulations! You've won $1,000,000! Click now to claim your prize before it ...
Prediction: 🚨 PHISHING (Confidence: 80.2%)

[Email 4]
Text: Meeting reminder: Project sync at 3pm tomorrow in Conference Room B....
Prediction: ✅ SAFE (Confidence: 66.7%)

[Email 5]
Text: Your PayPal account has unusual activity. Verify now: http://paypa1-secure.com/v...
Prediction: ✅ SAFE (Confidence: 50.2%)



In [ ]:
# Save the Model
# Create models directory
os.makedirs('models', exist_ok=True)

model_path = 'models/phishing_model.pkl'
joblib.dump(pipeline, model_path)

# Save model info
import json
model_info = {
    'model_path': model_path,
    'accuracy': float(accuracy),
    'roc_auc': float(roc_auc),
    'confusion_matrix': cm.tolist(),
    'cv_mean_accuracy': float(cv_scores.mean()),
    'training_samples': len(X_train),
    'test_samples': len(X_test)
}

with open('models/model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"✅ Model saved to: {model_path}")
print(f"✅ Model info saved to: models/model_info.json")
print(f"\n📦 Model size: {os.path.getsize(model_path) / (1024*1024):.2f} MB")

✅ Model saved to: models/phishing_model.pkl
✅ Model info saved to: models/model_info.json

📦 Model size: 20.50 MB


In [ ]:
# Download model files from Colab
import os

# First, verify the files exist
print(" Checking if files exist...")

if os.path.exists('models/phishing_model.pkl'):
    size = os.path.getsize('models/phishing_model.pkl') / (1024*1024)
    print(f" phishing_model.pkl exists ({size:.2f} MB)")
else:
    print(" phishing_model.pkl NOT FOUND!")
    print("   Run the 'Save the Model' cell first!")

if os.path.exists('models/model_info.json'):
    print(" model_info.json exists")
else:
    print(" model_info.json NOT FOUND!")

# List all files in models folder
print("\n Contents of models/ folder:")
if os.path.exists('models'):
    for f in os.listdir('models'):
        print(f"   - {f}")
else:
    print("   models/ folder doesn't exist!")

# Download files
print("\n Starting download...")
try:
    from google.colab import files
    
    if os.path.exists('models/phishing_model.pkl'):
        files.download('models/phishing_model.pkl')
        print("phishing_model.pkl download triggered!")
    
    if os.path.exists('models/model_info.json'):
        files.download('models/model_info.json')
        print(" model_info.json download triggered!")
    
    print("\n Check your browser's Downloads folder!")
    print("   If download didn't start, check for popup blockers.")
    
except ImportError:
    print(" Not running in Google Colab!")
    print("   Files are saved locally at:", os.path.abspath('models/'))
except Exception as e:
    print(f" Error: {e}")

📁 Checking if files exist...
✅ phishing_model.pkl exists (20.50 MB)
✅ model_info.json exists

📂 Contents of models/ folder:
   - model_info.json
   - phishing_model.pkl

📥 Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ phishing_model.pkl download triggered!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ model_info.json download triggered!

🎉 Check your browser's Downloads folder!
   If download didn't start, check for popup blockers.


In [1]:
# Download using Colab's file browser (SIMPLEST METHOD)
import os

print("="*60)
print("📥 DOWNLOAD YOUR MODEL FILES")
print("="*60)

# Check files exist
pkl_exists = os.path.exists('models/phishing_model.pkl')
json_exists = os.path.exists('models/model_info.json')

if pkl_exists:
    pkl_size = os.path.getsize('models/phishing_model.pkl') / (1024*1024)
    print(f"\n✅ phishing_model.pkl exists ({pkl_size:.2f} MB)")
else:
    print("\n❌ phishing_model.pkl NOT FOUND - run Save Model cell first!")

if json_exists:
    json_size = os.path.getsize('models/model_info.json') / 1024
    print(f"✅ model_info.json exists ({json_size:.2f} KB)")
else:
    print("❌ model_info.json NOT FOUND - run Save Model cell first!")

if pkl_exists and json_exists:
    print("\n" + "="*60)
    print("📁 HOW TO DOWNLOAD:")
    print("="*60)
    print("""
1. Look at the LEFT SIDEBAR in Colab
2. Click the 📁 FOLDER ICON (Files)
3. Navigate to: models/
4. Right-click on 'phishing_model.pkl' → Download
5. Right-click on 'model_info.json' → Download
    """)
    
    # Also try the files.download method
    print("="*60)
    print("🔄 Attempting auto-download...")
    print("="*60)
    
    try:
        from google.colab import files
        print("\nDownloading phishing_model.pkl...")
        files.download('models/phishing_model.pkl')
        print("Downloading model_info.json...")
        files.download('models/model_info.json')
        print("\n✅ Check your browser downloads!")
    except Exception as e:
        print(f"\n⚠️ Auto-download failed: {e}")
        print("\n👆 Use the MANUAL METHOD above (sidebar → files → download)")
else:
    print("\n❌ Cannot download - files don't exist!")
    print("   Run cells 1-11 first, then come back here.")

📥 DOWNLOAD YOUR MODEL FILES

✅ phishing_model.pkl exists (20.50 MB)
✅ model_info.json exists (0.30 KB)

📁 HOW TO DOWNLOAD:

1. Look at the LEFT SIDEBAR in Colab
2. Click the 📁 FOLDER ICON (Files)
3. Navigate to: models/
4. Right-click on 'phishing_model.pkl' → Download
5. Right-click on 'model_info.json' → Download
    
🔄 Attempting auto-download...



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ Check your browser downloads!
